# Colab Setup
Run this notebook first in every session. It clones the repo, installs dependencies, checks GPU, and mounts Drive for persistent storage of checkpoints/outputs.

In [1]:
# 1. Mount Google Drive (persists checkpoints/outputs across sessions)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2. Clone the repo (replace with your actual GitHub URL once created)
!git clone https://github.com/jrsheffie/Review-Summarization-LLM.git
%cd Review-Summarization-LLM

Cloning into 'Review-Summarization-LLM'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 36 (delta 0), reused 36 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 21.48 KiB | 7.16 MiB/s, done.
/content/Review-Summarization-LLM


In [3]:
# 3. Install dependencies
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 19.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.0 MB/s eta 0:00:00


In [4]:
# 4. Confirm GPU runtime is enabled (Runtime -> Change runtime type -> T4 GPU)
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- go to Runtime > Change runtime type > GPU')

CUDA available: True
Device: Tesla T4


In [6]:
import os
os.environ['KAGGLE_USERNAME'] = 'jsheffie'
os.environ['KAGGLE_KEY'] = 'KGAT_6307d48533c1425fe724f70bc0f3507d'

!python data/download_dataset.py

Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:01<00:00, 64.7MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [7]:
# 5. Set up Kaggle API for dataset download
# Upload your kaggle.json via the file browser on the left, then run:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!python data/download_dataset.py

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/arhamrumi/amazon-product-reviews
License(s): CC0-1.0
100% 115M/115M [00:00<00:00, 138MB/s]

Dataset downloaded to /content/Review-Summarization-LLM/data/raw
Next: check the downloaded CSV's column names against COLUMN_MAP in data/preprocess.py


In [10]:
import pandas as pd
df = pd.read_csv('data/raw/Reviews.csv', nrows=5)
print(df.columns.tolist())
df.head()

['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [11]:
# 6. Run the preprocessing pipeline
!python data/preprocess.py --input data/raw/Reviews.csv --output-dir data/processed

2026-07-18 18:54:08,003 [INFO] Loaded 568454 rows from data/raw/Reviews.csv
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'review_text'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/content/Review-Summarization-LLM/data/preprocess.py", line 257, in <module>
    run_pipeline(
  File "/content/Review-Summarization-LLM/data/preprocess.py", line 222, in run_pipeline
    df 

In [9]:
!ls data/raw/

Reviews.csv  sample_reviews.csv
